In [1]:
!test -f obs_3day.bufr \
    || wget https://sites.ecmwf.int/repository/pdbufr/test-data/obs_3day.bufr \
             --output-document=obs_3day.bufr

# Flat reader: filters

In [2]:
import datetime

import pdbufr

## Scalar filter – exact match

A scalar value selects messages where the key equals that value.  A key without a rank is treated as rank 1 (`"stationNumber"` → `"#1#stationNumber"`).

In [3]:
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude", "airTemperatureAt2M"],
    filters={"stationNumber": 894},
    reader="flat",
)
df

,ident,latitude,longitude,airTemperatureAt2M,stationNumber
0,03894,49.43,-2.6,282.4,894


## List filter – match any value

A list selects messages where the key matches *any* of the listed values.

In [4]:
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude"],
    filters={"stationNumber": [894, 103, 590]},
    reader="flat",
)
df

,ident,latitude,longitude,stationNumber
0,03894,49.43,-2.60,894
1,03590,52.12,0.96,590
2,07103,48.04,-4.73,103


## Slice filter – range selection

A `slice` object selects messages whose key value falls within the given range (using Python slice semantics).  The special `count` key counts messages from 1 in order of appearance in the file.

In [5]:
# Messages 2 to 5 (inclusive)
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude"],
    filters={"count": slice(2, 6)},
    reader="flat",
)
df

,ident,latitude,longitude
0,03590,52.12,0.96
1,03379,53.03,-0.50
2,03391,53.09,-0.17
3,03743,51.20,-1.81
4,03749,51.15,-1.57


A slice with only a start value selects from that position to the end of the file:

In [6]:
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude"],
    filters={"count": slice(48, None)},
    reader="flat",
)
df

,ident,latitude
0,07029,49.35
1,07103,48.04
2,03008,59.53


## Ranked-key filter

The `#N#` rank prefix selects a specific occurrence of a repeated key.  The example below matches messages where the **first** cloud layer (`#1#cloudType`) is type 35 *and* the **second** layer (`#2#cloudType`) is type 27.  Filter keys are automatically added to the output columns.

In [7]:
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude"],
    filters={"#1#cloudType": 35, "#2#cloudType": 27},
    reader="flat",
)
df

,ident,latitude,longitude,#1#cloudType,#2#cloudType
0,07149,48.72,2.38,35,27
1,07145,48.77,2.01,35,27


## Tilde filter (`~`) – match any occurrence

The leading `~` selects messages where *any* occurrence of the key matches.  This is useful when you do not know (or do not care) which rank holds the value.

In [8]:
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude"],
    filters={"~cloudType": 62},
    reader="flat",
)
df

,ident,latitude,longitude
0,03105,55.68,-6.25
1,06252,53.22,3.22
2,06310,51.44,3.60


## Datetime filter

Datetime keys can be filtered with a `datetime` value or a slice of datetimes. The computed key `typical_datetime` combines the individual date and time header fields into a single `datetime` object:

In [9]:
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "typical_datetime", "latitude", "longitude"],
    filters={"typical_datetime": datetime.datetime(2017, 4, 25, 12, 0)},
    reader="flat",
)
print(f"{len(df)} rows")
df.head()

50 rows


,ident,typical_datetime,latitude,longitude
0,03894,2017-04-25 12:00:00,49.43,-2.60
1,03590,2017-04-25 12:00:00,52.12,0.96
2,03379,2017-04-25 12:00:00,53.03,-0.50
3,03391,2017-04-25 12:00:00,53.09,-0.17
4,03743,2017-04-25 12:00:00,51.20,-1.81


## Computed-key filter

Computed keys can be used in filters.  `WMO_station_id = blockNumber * 1000 + stationNumber` so a value of 3894 corresponds to block 3, station 894:

In [10]:
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "WMO_station_id", "latitude", "longitude"],
    filters={"WMO_station_id": [3894, 7103]},
    reader="flat",
)
df

,ident,WMO_station_id,latitude,longitude
0,03894,3894,49.43,-2.60
1,07103,7103,48.04,-4.73


## Combining filters

Multiple conditions are combined with logical AND.  The example below selects messages from a specific rdbtimeTime *and* stationNumber:

In [11]:
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "rdbtimeTime", "latitude", "longitude", "airTemperatureAt2M"],
    filters={"rdbtimeTime": "115557", "dataSubCategory": 1},
    reader="flat",
)
print(f"{len(df)} rows")
df.head()

6 rows


,ident,rdbtimeTime,latitude,longitude,airTemperatureAt2M,dataSubCategory
0,03761,115557,51.24,-0.94,280.9,1
1,03105,115557,55.68,-6.25,NaN,1
2,03895,115557,49.21,-2.19,282.8,1
3,03068,115557,57.71,-3.32,276.6,1
4,03809,115557,50.08,-5.26,282.5,1
